# 🔗 PySpark Joins — Comprehensive Training Notebook

**Author:** Raamashaamy  
**Platform:** Databricks (Community Edition / Any Cluster)  
**Audience:** Freshers / Beginners  

---

## 📋 Topics Covered

| # | Topic |
|---|-------|
| 1 | Dataset Setup — Employees & Departments |
| 2 | Inner Join |
| 3 | Left Join (Left Outer) |
| 4 | Right Join (Right Outer) |
| 5 | Full Outer Join |
| 6 | `.join(df2, on=..., how=...)` Syntax Explained |
| 7 | Joining on Multiple Columns |
| 8 | Handling Duplicate Column Names After a Join |
| 9 | Broadcast Join — What It Is & When to Use It |

---

> 💡 **Tip:** Run each cell top-to-bottom. Every section has explanations before the code and observations after.

---
## 📂 Section 1 — Dataset Setup

We will use **three small, relatable datasets** throughout this notebook:

| Dataset | Description |
|---------|-------------|
| `employees_df` | Employee details — emp_id, name, dept_id, city |
| `departments_df` | Department details — dept_id, dept_name, location |
| `salaries_df` | Salary info — emp_id, dept_id, salary (for multi-column join demo) |

These datasets are **intentionally designed** so that:
- Some employees have **no matching department** (to demonstrate LEFT join behaviour).
- Some departments have **no employees** (to demonstrate RIGHT join behaviour).
- The FULL OUTER join captures **all** unmatched rows from both sides.

In [0]:
from pyspark.sql import functions as F

# ── Employees DataFrame ─────────────────────────────────────────────────────
# dept_id = 99  → does NOT exist in departments_df (intentional for LEFT join demo)
employees_data = [
    (101, "Alice",   10, "Chennai"),
    (102, "Bob",     20, "Bangalore"),
    (103, "Charlie", 10, "Chennai"),
    (104, "Diana",   30, "Mumbai"),
    (105, "Eve",     99, "Delhi"),    # dept_id 99 → no matching dept
    (106, "Frank",   20, "Bangalore"),
]
emp_schema = ["emp_id", "name", "dept_id", "city"]
employees_df = spark.createDataFrame(employees_data, emp_schema)

# ── Departments DataFrame ────────────────────────────────────────────────────
# dept_id = 40 → has no employees (intentional for RIGHT join demo)
departments_data = [
    (10, "Engineering",  "Chennai"),
    (20, "Marketing",    "Bangalore"),
    (30, "Finance",      "Mumbai"),
    (40, "HR",           "Hyderabad"),  # dept 40 → no employees assigned
]
dept_schema = ["dept_id", "dept_name", "location"]
departments_df = spark.createDataFrame(departments_data, dept_schema)

# ── Salaries DataFrame (for multi-column join) ───────────────────────────────
# Uses BOTH emp_id AND dept_id as join keys
salaries_data = [
    (101, 10, 75000),
    (102, 20, 82000),
    (103, 10, 68000),
    (104, 30, 91000),
    (105, 99, 55000),   # matches employees_df but not departments_df
    (107, 20, 61000),   # emp_id 107 doesn't exist in employees_df
]
sal_schema = ["emp_id", "dept_id", "salary"]
salaries_df = spark.createDataFrame(salaries_data, sal_schema)

print("=" * 55)
print("📋 employees_df")
print("=" * 55)
employees_df.show()

print("=" * 55)
print("📋 departments_df")
print("=" * 55)
departments_df.show()

print("=" * 55)
print("📋 salaries_df")
print("=" * 55)
salaries_df.show()

---
## 🔑 Section 2 — The `.join()` Syntax Explained

```python
result_df = df1.join(df2, on=<condition>, how=<join_type>)
```

| Parameter | What it does | Example |
|-----------|-------------|--------|
| `df2` | The **right** DataFrame to join with | `departments_df` |
| `on` | Join **condition** — column name (string), list of names, or boolean expression | `"dept_id"` or `["emp_id", "dept_id"]` |
| `how` | Type of join | `"inner"`, `"left"`, `"right"`, `"outer"` |

### Two ways to write the `on` condition:

```python
# ✅ Approach 1 — Column name as STRING (recommended when column names match in both DFs)
df1.join(df2, on="dept_id", how="inner")

# ✅ Approach 2 — Boolean expression (use when column names differ OR for complex conditions)
df1.join(df2, on=df1["dept_id"] == df2["dept_id"], how="inner")
```

> ⚠️ **Important:**  
> When you use the **string approach** (`on="dept_id"`), the result will have **only one** `dept_id` column.  
> When you use the **expression approach** (`on=df1["dept_id"] == df2["dept_id"]`), **both** `dept_id` columns appear (causing duplicates). We'll handle that in Section 8.

---
## 🔵 Section 3 — INNER JOIN

### Concept
An **INNER JOIN** returns only the rows where there is a **match in BOTH DataFrames**.

In [0]:
inner_df = employees_df.join(departments_df, on="dept_id", how="inner")
inner_df.show()

---
## 🟡 Section 4 — LEFT JOIN (Left Outer Join)

### Concept
A **LEFT JOIN** returns:
- **ALL rows** from the **left** DataFrame (`employees_df`)
- Matching rows from the right DataFrame (`departments_df`)
- If there's **no match**, the right-side columns are filled with **`null`**

**Use case:** "Show me ALL employees — even those not yet assigned to a department."

In [0]:

left_df = employees_df.join(departments_df, on="dept_id", how="left")
left_df.show()

left_df.filter(F.col("dept_name").isNull()).show()
print("💡 Tip: Filter on a right-side column being NULL to find unmatched rows.")

---
## 🟠 Section 5 — RIGHT JOIN (Right Outer Join)

### Concept
A **RIGHT JOIN** is the mirror of LEFT JOIN:
- **ALL rows** from the **right** DataFrame (`departments_df`) are kept
- Matching rows from the left DataFrame (`employees_df`) are included
- If no match, left-side columns are **`null`**

**Use case:** "Show me ALL departments — even those that have no employees yet."

In [0]:

right_df = employees_df.join(departments_df, on="dept_id", how="right")
right_df.orderBy("dept_id", "emp_id").show()

print("--- 🔍 Filter: Departments with NO employees ---")
right_df.filter(F.col("emp_id").isNull()).show()
print("💡 Tip: A RIGHT JOIN is same as swapping the DataFrames in a LEFT JOIN.")

---
## 🔴 Section 6 — FULL OUTER JOIN

### Concept
A **FULL OUTER JOIN** (also called `outer` in PySpark) keeps **ALL rows from BOTH sides**:
- Matched rows → columns from both sides filled
- Unmatched from LEFT → right-side columns are `null`
- Unmatched from RIGHT → left-side columns are `null`

```
Result = INNER JOIN  +  LEFT-only unmatched rows  +  RIGHT-only unmatched rows
```

**Use case:** Data reconciliation — "Show me everything from both sides, including mismatches."

In [0]:
full_df = employees_df.join(departments_df, on="dept_id", how="outer")
full_df.orderBy("dept_id", "emp_id").show()

print("--- 🔍 All unmatched rows from BOTH sides ---")
full_df.filter(F.col("dept_name").isNull() | F.col("emp_id").isNull()).show()

---
## 📊 Section 7 — Quick Comparison: All Four Join Types

| Join Type | Left Unmatched Rows | Right Unmatched Rows | PySpark `how=` values |
|-----------|:-------------------:|:--------------------:|----------------------|
| INNER | ❌ Dropped | ❌ Dropped | `"inner"` |
| LEFT OUTER | ✅ Kept (right = null) | ❌ Dropped | `"left"` or `"left_outer"` |
| RIGHT OUTER | ❌ Dropped | ✅ Kept (left = null) | `"right"` or `"right_outer"` |
| FULL OUTER | ✅ Kept (right = null) | ✅ Kept (left = null) | `"outer"`, `"full"`, or `"full_outer"` |

In [0]:
print("📊 Row count comparison across all join types:\n")

summary_data = [
    ("INNER",      inner_df.count()),
    ("LEFT",       left_df.count()),
    ("RIGHT",      right_df.count()),
    ("FULL OUTER", full_df.count()),
]

summary_df = spark.createDataFrame(summary_data, ["Join Type", "Row Count"])
summary_df.show(truncate=False)

---
## 🔑 Section 8 — Joining on MULTIPLE Columns

### When do you need a multi-column join?
When a single column is **not enough** to uniquely identify a match.  
For example: an employee can work in **multiple departments over time** — so you need both `emp_id` AND `dept_id` to find the right salary record.

### Syntax
```python
# Pass a LIST of column names to the `on` parameter
df1.join(df2, on=["col1", "col2"], how="inner")
```

We will join `employees_df` with `salaries_df` on **both** `emp_id` and `dept_id`.

In [0]:
# INNER multi-column join
multi_join_df = employees_df.join(
    salaries_df,
    on=["emp_id", "dept_id"],   # ← LIST of keys
    how="inner"
)
multi_join_df.show()

# LEFT multi-column join — to see unmatched employees
print("\n--- LEFT JOIN version (keep all employees) ---")
employees_df.join(salaries_df, on=["emp_id", "dept_id"], how="left").show()

---
## ⚠️ Section 9 — Handling Duplicate Column Names After a Join

### The Problem
When you join using a **boolean expression** (instead of a string), PySpark keeps **both copies** of the join key column, causing ambiguity.

```python
# This creates TWO dept_id columns:
df = employees_df.join(departments_df,
         on=employees_df["dept_id"] == departments_df["dept_id"],
         how="inner")
df.select("dept_id")  # ❌  AnalysisException: Ambiguous reference to dept_id
```

### Three Ways to Fix It

| # | Solution | When to Use |
|---|----------|-------------|
| 1 | Use **string** `on="dept_id"` | Best when column names are the same in both DFs |
| 2 | **Drop** the duplicate column after join | When you used a boolean expression |
| 3 | Use **aliases** on DataFrames + fully qualified names | When column names differ or complex joins |

In [0]:
# ── Step 1: Reproduce the problem ───────────────────────────────────────────
print("\n🔴 Step 1: Join using BOOLEAN expression → creates duplicate dept_id")
dup_df = employees_df.join(
    departments_df,
    on=employees_df["dept_id"] == departments_df["dept_id"],  # expression form
    how="inner"
)
print("Columns in dup_df:", dup_df.columns)
dup_df.show()

# Trying to select dept_id will raise AnalysisException — we catch it safely:
print("\n🔴 Trying dup_df.select('dept_id') → causes ambiguity error:")
try:
    dup_df.select("dept_id").show()
except Exception as e:
    print(f"   ❌ Error caught: {type(e).__name__}")
    print(f"   Message: {str(e)[:120]}...")

In [0]:
# String form keeps only ONE dept_id column automatically
fix1_df = employees_df.join(departments_df, on="dept_id", how="inner")
print("Columns in fix1_df:", fix1_df.columns)
print("✅ Only ONE dept_id — no ambiguity!")
fix1_df.select("dept_id", "name", "dept_name").show()

In [0]:

fix2_df = employees_df.join(
    departments_df,
    on=employees_df["dept_id"] == departments_df["dept_id"],
    how="inner"
).drop(departments_df["dept_id"])   # ← drop the RIGHT table matching column

print("Columns in fix2_df:", fix2_df.columns)
print("✅ Duplicate removed — one dept_id remains!")
fix2_df.select("dept_id", "name", "dept_name").show()

In [0]:

# Alias both DataFrames
emp_aliased  = employees_df.alias("emp")
dept_aliased = departments_df.alias("dept")

fix3_df = emp_aliased.join(
    dept_aliased,
    on=F.col("emp.dept_id") == F.col("dept.dept_id"),
    how="inner"
).select(
    F.col("emp.emp_id"),
    F.col("emp.name"),
    F.col("emp.dept_id").alias("dept_id"),   # pick ONE, rename it
    F.col("dept.dept_name"),
    F.col("dept.location")
)

print("Columns:", fix3_df.columns)
print("✅ Fully qualified selection — clean output!")
fix3_df.show()

---
## 📡 Section 10 — BROADCAST JOIN

### What Is a Broadcast Join?

In a regular Spark join, both DataFrames are **shuffled** across the network — every partition that needs to match sends data to the right node. This is expensive.

A **Broadcast Join** avoids the shuffle by sending an **entire copy** of the **small DataFrame** to **every executor** in memory. Each executor then performs the join locally — much faster!

```
     ┌─────────────────────────────────────────────┐
     │           NORMAL JOIN (Shuffle Merge)         │
     │                                               │
     │  Partition 1 ──►  shuffle ──►  join node 1   │
     │  Partition 2 ──►  shuffle ──►  join node 2   │
     │  Partition 3 ──►  shuffle ──►  join node 3   │
     │   (network shuffle = SLOW for large data)     │
     └─────────────────────────────────────────────┘

     ┌─────────────────────────────────────────────┐
     │           BROADCAST JOIN                     │
     │                                              │
     │  small_df → broadcast to ALL executors 📡    │
     │                                              │
     │  Partition 1 ──►  local join (no shuffle!)   │
     │  Partition 2 ──►  local join (no shuffle!)   │
     │  Partition 3 ──►  local join (no shuffle!)   │
     └─────────────────────────────────────────────┘
```

### When to Use `broadcast()`?

| Condition | Use Broadcast? |
|-----------|:--------------:|
| One DataFrame is **small** (< ~10 MB default threshold) | ✅ Yes |
| Joining a **dimension table** (e.g., departments, country codes) with a **large fact table** | ✅ Yes |
| Both DataFrames are **large** (millions of rows each) | ❌ No — use shuffle join |
| Small DF is **frequently re-used** in multiple joins | ✅ Yes — broadcast it once |

### How to Use
```python
from pyspark.sql.functions import broadcast

result = large_df.join(broadcast(small_df), on="key", how="inner")
```

> 💡 **Auto-broadcast threshold:** Spark automatically broadcasts any DF under `spark.sql.autoBroadcastJoinThreshold` bytes (default: **10 MB**). You can set this to `-1` to disable auto-broadcast.

In [0]:
from pyspark.sql.functions import broadcast

# departments_df is the small lookup table → broadcast it
broadcast_df = employees_df.join(
    broadcast(departments_df),   # ← wrap small DF with broadcast()
    on="dept_id",
    how="inner"
)

broadcast_df.show()

In [0]:
regular_join = employees_df.join(departments_df, on="dept_id", how="inner")
broadcast_join = employees_df.join(broadcast(departments_df), on="dept_id", how="inner")

print("\n--- Regular Join Plan ---")
regular_join.explain()

print("\n--- Broadcast Join Plan ---")
broadcast_join.explain()

print()
print("📌 What to look for in the plan:")
print("   • Regular join  → look for 'SortMergeJoin' or 'ShuffledHashJoin'")
print("   • Broadcast join → look for 'BroadcastHashJoin' or 'BroadcastExchange'")
print("   → No shuffle stage in the broadcast plan = faster execution!")

---
## ✅ Section 11 — Complete Recap

### PySpark Join Syntax
```python
result = df1.join(df2, on="col",         how="inner")    # single key
result = df1.join(df2, on=["c1","c2"],   how="left")     # multi-key
result = df1.join(broadcast(df2), on="col", how="inner") # broadcast
```

### Join Type Cheat Sheet
| `how=` value | What it keeps |
|---|---|
| `"inner"` | Only matching rows from both sides |
| `"left"` or `"left_outer"` | All left rows + matched right (nulls if no match) |
| `"right"` or `"right_outer"` | All right rows + matched left (nulls if no match) |
| `"outer"`, `"full"`, `"full_outer"` | All rows from both sides |

### Duplicate Column Fixes
1. **Preferred:** Use `on="col_name"` (string) — keeps only one copy
2. `.drop(df2["col"])` — drop the duplicate after join
3. `df.alias("a")` + `F.col("a.col")` — qualify with alias

### Broadcast Join
- Wrap the **small** DataFrame: `broadcast(small_df)`
- Avoids shuffle → faster joins with dimension tables
- Auto-triggered for DFs < 10 MB (configurable)